# Lab 2 - Task 1: Frequency-Based Baseline

Baseline con frequenze delle API call e Logistic Regression.

## Funzioni

In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

In [ ]:
RANDOM_STATE = 42
TRAIN_PATH = Path("train.json")
TEST_PATH = Path("test.json")


def load_dataset(path):
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    X = [row["api_call_sequence"] for row in data]
    y = np.array([row["is_malware"] for row in data], dtype=int)
    return X, y


def vocabulary(sequences):
    return set(api for seq in sequences for api in seq)


def sequence_length_summary(sequences):
    lengths = np.array([len(seq) for seq in sequences])
    return {
        "n_samples": len(sequences),
        "min_len": int(lengths.min()),
        "median_len": float(np.median(lengths)),
        "mean_len": float(lengths.mean()),
        "max_len": int(lengths.max()),
    }


def dataset_overview(X_train, y_train, X_test, y_test):
    rows = []
    for split, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
        label_counts = Counter(y)
        row = sequence_length_summary(X)
        row.update({
            "split": split,
            "goodware_0": label_counts.get(0, 0),
            "malware_1": label_counts.get(1, 0),
            "malware_%": 100 * label_counts.get(1, 0) / len(y),
            "unique_api_calls": len(vocabulary(X)),
        })
        rows.append(row)
    cols = ["split", "n_samples", "goodware_0", "malware_1", "malware_%", "min_len", "median_len", "mean_len", "max_len", "unique_api_calls"]
    return pd.DataFrame(rows)[cols].round(3)


def vocabulary_report(X_train, X_test):
    train_vocab = vocabulary(X_train)
    test_vocab = vocabulary(X_test)
    only_test = sorted(test_vocab - train_vocab)
    only_train = sorted(train_vocab - test_vocab)
    summary = pd.DataFrame([{
        "train_vocab_size": len(train_vocab),
        "test_vocab_size": len(test_vocab),
        "intersection": len(train_vocab & test_vocab),
        "only_in_test_count": len(only_test),
        "only_in_train_count": len(only_train),
    }])
    return summary, only_test, only_train


def sequences_to_text(sequences):
    return [" ".join(seq) for seq in sequences]


def build_frequency_matrices(X_train, X_test):
    train_vocab = sorted(vocabulary(X_train))
    vectorizer = CountVectorizer(
        analyzer="word",
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        vocabulary=train_vocab,
        lowercase=False,
    )
    X_train_freq = vectorizer.fit_transform(sequences_to_text(X_train))
    X_test_freq = vectorizer.transform(sequences_to_text(X_test))
    feature_names = vectorizer.get_feature_names_out()
    return X_train_freq, X_test_freq, feature_names, vectorizer


def sparsity_report(X_train_freq, X_test_freq):
    rows = []
    for split, matrix in [("train", X_train_freq), ("test", X_test_freq)]:
        nnz_per_row = matrix.getnnz(axis=1)
        n_features = matrix.shape[1]
        rows.append({
            "split": split,
            "shape": f"{matrix.shape[0]} x {matrix.shape[1]}",
            "avg_non_zero_per_row": nnz_per_row.mean(),
            "min_non_zero": nnz_per_row.min(),
            "median_non_zero": np.median(nnz_per_row),
            "max_non_zero": nnz_per_row.max(),
            "non_zero_ratio": nnz_per_row.mean() / n_features,
            "zero_ratio": 1 - (nnz_per_row.mean() / n_features),
        })
    return pd.DataFrame(rows).round(4)


def train_logistic_regression(X_train_freq, y_train):
    pipe = Pipeline([
        ("clf", LogisticRegression(solver="liblinear", max_iter=2000, random_state=RANDOM_STATE)),
    ])
    param_grid = {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__class_weight": [None, "balanced"],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(pipe, param_grid=param_grid, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True)
    search.fit(X_train_freq, y_train)
    cv_results = pd.DataFrame(search.cv_results_)[
        ["param_clf__C", "param_clf__class_weight", "mean_test_score", "std_test_score", "rank_test_score"]
    ].sort_values("rank_test_score")
    return search.best_estimator_, search.best_params_, search.best_score_, cv_results


def evaluate_classifier(model, X_test_freq, y_test):
    y_pred = model.predict(X_test_freq)
    y_score = model.predict_proba(X_test_freq)[:, 1]
    metrics = pd.DataFrame([{
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_goodware_0": f1_score(y_test, y_pred, pos_label=0, zero_division=0),
        "f1_malware_1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
    }]).round(4)
    cm = pd.DataFrame(
        confusion_matrix(y_test, y_pred, labels=[0, 1]),
        index=["true_goodware_0", "true_malware_1"],
        columns=["pred_goodware_0", "pred_malware_1"],
    )
    report = pd.DataFrame(classification_report(y_test, y_pred, labels=[0, 1], target_names=["goodware_0", "malware_1"], output_dict=True, zero_division=0)).T.round(4)
    return metrics, cm, report


def api_presence_in_test_only(X_test, only_test_apis):
    counts = Counter(api for seq in X_test for api in seq if api in only_test_apis)
    rows = [{"api_call": api, "test_frequency": counts[api]} for api in only_test_apis]
    return pd.DataFrame(rows)

## Output per rispondere al Task 1

In [ ]:
X_train, y_train = load_dataset(TRAIN_PATH)
X_test, y_test = load_dataset(TEST_PATH)

overview = dataset_overview(X_train, y_train, X_test, y_test)
display(overview)

In [ ]:
vocab_summary, only_test_apis, only_train_apis = vocabulary_report(X_train, X_test)
display(vocab_summary)
print("API call presenti solo nel test:")
print(only_test_apis)
display(api_presence_in_test_only(X_test, only_test_apis))

In [ ]:
X_train_freq, X_test_freq, feature_names, vectorizer = build_frequency_matrices(X_train, X_test)
print(f"Feature usate: vocabolario del train ({len(feature_names)} colonne).")
print("Le API presenti solo nel test vengono ignorate perch? non hanno una colonna appresa dal train.")
display(sparsity_report(X_train_freq, X_test_freq))

In [ ]:
model, best_params, best_cv_score, cv_results = train_logistic_regression(X_train_freq, y_train)
print("Migliori iperparametri:", best_params)
print(f"Miglior F1 macro medio in 5-fold CV: {best_cv_score:.4f}")
display(cv_results.round(4))

In [ ]:
metrics, cm, report = evaluate_classifier(model, X_test_freq, y_test)
print("Metriche finali sul test set")
display(metrics)
print("Matrice di confusione")
display(cm)
print("Classification report")
display(report)

In [ ]:
answers = [
    ["Vocabolario", f"Train: {len(vocabulary(X_train))} API uniche. Test: {len(vocabulary(X_test))} API uniche. Solo nel test: {len(only_test_apis)} ({', '.join(only_test_apis)})."],
    ["Costruzione test dataframe", "Uso sempre il vocabolario del train per train e test. Le API solo nel test sono out-of-vocabulary e vengono ignorate, evitando leakage dal test."],
    ["Sparsita", "Ogni riga ha 258 feature. La tabella sopra mostra gli elementi non-zero medi e il rapporto rispetto al numero di feature."],
    ["Ordine", "No. Il dataframe frequency-based conserva solo quante volte compare ogni API call, non la posizione: non sappiamo piu quale API sia arrivata prima."],
    ["Classifier", f"Logistic Regression. Ho scelto C e class_weight con GridSearchCV 5-fold sul train, ottimizzando F1 macro. Parametri finali: {best_params}."],
    ["Qualita finale", "La performance e buona se accuracy/F1 malware sono alti, ma va letta insieme a balanced accuracy e F1 goodware perche il dataset e fortemente sbilanciato verso malware."],
]
display(pd.DataFrame(answers, columns=["Domanda", "Risposta sintetica"]))